# Classical Speed-Density Models: Validation Benchmark for SR-210 Mesa ABM

This notebook encodes the three canonical macroscopic traffic flow models — **Greenshields (1935)**, **Greenberg (1959)**, and **Underwood (1961)** — using parameter values drawn from published empirical studies. It serves three purposes:

1. **Encode and visualize** the three models with literature-derived parameter ranges, including sensitivity analysis across physically plausible parameter values.
2. **Characterize parameter uncertainty** via Monte Carlo sampling across cross-study estimates, producing publication-quality figures with 95% CI bands.
3. **Provide the ABM comparison harness**: once Tier 2 trajectory data from the SR-210 Mesa simulation is available, this notebook extracts the macroscopic fundamental diagram via Edie's generalized definitions and fits the three models via nonlinear least squares.

**No published study has calibrated these models on a steep two-lane mountain road.** The literature spans freeways (Drake 1967), urban arterials (Lu & Meng 2013), expressways (Romanowska 2021), and one two-lane road (Tiwari 2014 Nepal). SR-210 is distinct: 12.4 miles, grades up to 11%, 25–50 mph speed limits, seasonal closure patterns. This notebook produces what may be the first fundamental diagram characterization of a facility like SR-210.

In [ ]:
import os
import subprocess
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import norm

warnings.filterwarnings('ignore', category=RuntimeWarning)
np.random.seed(42)

# --- project root ---
PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

# --- output directory ---
Path("notebooks/figures").mkdir(exist_ok=True)

# --- style constants ---
FIG_W_SINGLE = (10, 6)
FIG_W_WIDE   = (14, 6)
FIG_W_TRIPLE = (18, 5)
GRID_ALPHA   = 0.3
CI_ALPHA     = 0.20
N_MC         = 2000
CI_LO, CI_HI = 2.5, 97.5

# --- density axis for all plots ---
K_PLOT  = np.linspace(0.5, 160, 600)   # veh/mi (start at 0.5 to avoid Greenberg log singularity)
K_RANGE = np.linspace(0.5, 160, 500)   # coarser version for sensitivity sweeps

sns.set_style("whitegrid")
print("Setup complete.")

---
## 1. Unit Conversions

The five literature sources span two unit systems: **imperial** (Drake 1967 uses mph and veh/mi) and **metric** (Lu & Meng 2013, Romanowska 2021, and Tiwari 2014 use km/h and veh/km). All parameter values are normalized to **mph** and **veh/mi** for consistency with the Mesa ABM output, which reports speed in mph and can produce density in veh/mi from trajectory data.

Conversions are defined inline so this notebook is fully self-contained — no project imports required for this section.

In [ ]:
# --- conversion constants ---
KMH_TO_MPH    = 0.621371       # km/h → mph
VEHKM_TO_VEHMI = 1.60934       # veh/km → veh/mi (multiply by 1.60934)
METERS_PER_MILE = 1609.34

def kmh_to_mph(v):   return v * KMH_TO_MPH
def mph_to_kmh(v):   return v / KMH_TO_MPH
def vehkm_to_vehmi(k): return k * VEHKM_TO_VEHMI
def vehmi_to_vehkm(k): return k / VEHKM_TO_VEHMI

# --- validation table ---
print("Unit conversion validation:")
print(f"  Tiwari 62.9 km/h  →  {kmh_to_mph(62.9):.1f} mph  (expect ~39.1)")
print(f"  Tiwari 150 veh/km →  {vehkm_to_vehmi(150):.1f} veh/mi  (expect ~241.4)")
print(f"  Lu&Meng 80.3 km/h →  {kmh_to_mph(80.3):.1f} mph  (expect ~49.9)")
print(f"  Lu&Meng 114.2 veh/km → {vehkm_to_vehmi(114.2):.1f} veh/mi  (expect ~183.8)")
print(f"  Roman 121 km/h    →  {kmh_to_mph(121):.1f} mph  (expect ~75.2)")
print(f"  Tiwari k_j=1452 veh/km → {vehkm_to_vehmi(1452):.0f} veh/mi  (outlier: expect ~2338)")

---
## 2. Model Equations

The three models form a natural historical trilogy (1935–1959–1961) that together span the full density range through **complementary failures**:

| Model | Equation | Parameters | v → v_f at k=0? | v → 0 at finite k_j? |
|---|---|---|---|---|
| **Greenshields (1935)** | $v = v_f\left(1 - \frac{k}{k_j}\right)$ | $v_f$, $k_j$ | ✓ Yes | ✓ Yes |
| **Greenberg (1959)** | $v = v_0 \ln\left(\frac{k_j}{k}\right)$ | $v_0$, $k_j$ | ✗ No ($v \to \infty$) | ✓ Yes |
| **Underwood (1961)** | $v = v_f \exp\left(-\frac{k}{k_0}\right)$ | $v_f$, $k_0$ | ✓ Yes | ✗ No ($k_j \to \infty$) |

**Origins:**
- Greenshields derived the linear model from ~7 data points collected with 16mm film cameras (1935, Highway Research Board Proc. Vol. 14). Simple OLS fitting applies directly.
- Greenberg derived the logarithmic model from a hydrodynamic analogy treating traffic as compressible fluid, calibrated on Lincoln Tunnel data (*Operations Research*, 1959, Vol. 7). It diverges as $k \to 0$ — a physical singularity requiring `np.clip(k, 1e-6, None)` in code.
- Underwood derived the exponential from Merritt Parkway data (Yale Bureau of Highway Traffic, 1961) — a limited-access two-lane facility, the closest road-type analog to SR-210 among the three origins.

Each model implies a parabolic flow-density curve $q = kv$ with a single capacity peak. Critical-point derivations:
- **Greenshields:** $k_c = k_j/2$, $q_{\max} = v_f k_j / 4$
- **Greenberg:** $k_c = k_j/e$, $q_{\max} = v_0 k_j / e$
- **Underwood:** $k_c = k_0$, $q_{\max} = v_f k_0 / e$

In [ ]:
# --- model functions ---

def greenshields(k, v_f, k_j):
    """Greenshields (1935): linear speed-density.
    k, k_j in veh/mi; v_f in mph. Returns speed in mph.
    """
    return v_f * (1.0 - k / k_j)


def greenberg(k, v_0, k_j):
    """Greenberg (1959): logarithmic speed-density.
    Undefined at k=0 — clipped to 1e-6 to avoid log singularity.
    Also clipped to k_j to avoid log of negative inside region k > k_j.
    k, k_j in veh/mi; v_0 in mph. Returns speed in mph.
    """
    k_safe = np.clip(np.asarray(k, dtype=float), 1e-6, k_j)
    return v_0 * np.log(k_j / k_safe)


def underwood(k, v_f, k_0):
    """Underwood (1961): exponential speed-density.
    k, k_0 in veh/mi; v_f in mph. Returns speed in mph.
    """
    return v_f * np.exp(-np.asarray(k, dtype=float) / k_0)


# --- derived critical-point functions ---

def greenshields_critical(v_f, k_j):
    """Returns (k_c veh/mi, q_max veh/hr)."""
    k_c   = k_j / 2.0
    q_max = v_f * k_j / 4.0
    return k_c, q_max


def greenberg_critical(v_0, k_j):
    """Returns (k_c veh/mi, q_max veh/hr)."""
    k_c   = k_j / np.e
    q_max = v_0 * k_c
    return k_c, q_max


def underwood_critical(v_f, k_0):
    """Returns (k_c veh/mi, q_max veh/hr)."""
    k_c   = k_0
    q_max = v_f * k_0 / np.e
    return k_c, q_max


# --- quick validation ---
k_test = np.array([1.0, 10.0, 50.0, 100.0, 125.0])
print("Validation (Drake 1967 Eisenhower params: GS v_f=58.6 k_j=125, GR v_0=32.8 k_j=146, UW v_f=76.8 k_0=56.9):")
print(f"{'k (veh/mi)':>12}  {'Greenshields':>13}  {'Greenberg':>12}  {'Underwood':>12}")
print("-" * 55)
for k in k_test:
    gs = greenshields(k, 58.6, 125)
    gr = greenberg(k,   32.8, 146)
    uw = underwood(k,   76.8, 56.9)
    print(f"{k:>12.1f}  {gs:>13.2f}  {gr:>12.2f}  {uw:>12.2f}")

---
## 3. Literature Parameter Registry

Five published studies provide calibrated parameter values for the three models. Each is described below:

- **Drake et al. (1967)** — Eisenhower Expressway, Chicago (3-lane, 55 mph). The canonical benchmark dataset for comparing macroscopic models; 118 one-minute observations. Already in imperial units. Most widely cited parameter comparison in the literature.
- **Lu & Meng (2013)** — Two sites in China: Beijing Third Ring Road (6-lane urban, 80 km/h) and Jing Jin Tang Highway (4-lane intercity, 110 km/h). Higher free-flow speeds than SR-210 but valuable for cross-study spread.
- **Romanowska & Jamroz (2021)** — S6 Expressway, Poland (4-lane, 120 km/h). Largest dataset (37.5M vehicles, 36 months). Provides Greenshields and Underwood only (excluded Greenberg due to boundary condition failure).
- **Tiwari & Marsani (2014)** — Nepal hilly terrain (two-lane undivided, mixed traffic). **Closest road-type analog to SR-210** — two-lane, steep, curved. However, the Greenberg $k_j = 1452$ veh/km (→ ~2338 veh/mi after conversion) is a known outlier, almost certainly caused by mixed-traffic counting that includes non-motorized vehicles, inflating observed density far beyond what a US highway would produce.

All metric values are converted to mph and veh/mi at registry time.

In [ ]:
# --- raw literature registry (before unit conversion) ---
LITERATURE_RAW = {
    "greenshields": [
        {"source": "Drake_1967_Eisenhower",   "v_f": 58.6,  "k_j": 125.0, "units": "imperial"},
        {"source": "Lu_Meng_2013_Beijing",     "v_f": 80.3,  "k_j": 114.2, "units": "metric"},
        {"source": "Lu_Meng_2013_JJT",         "v_f": 118.6, "k_j": 54.1,  "units": "metric"},
        {"source": "Romanowska_2021_S6",        "v_f": 121.0, "k_j": 138.0, "units": "metric"},
        {"source": "Tiwari_2014_Nepal",         "v_f": 62.9,  "k_j": 150.0, "units": "metric"},
    ],
    "greenberg": [
        {"source": "Drake_1967_Eisenhower",   "v_0": 32.8,  "k_j": 146.0,  "units": "imperial"},
        {"source": "Lu_Meng_2013_Beijing",     "v_0": 30.8,  "k_j": 143.9,  "units": "metric"},
        {"source": "Lu_Meng_2013_JJT",         "v_0": 47.0,  "k_j": 83.1,   "units": "metric"},
        {"source": "Tiwari_2014_Nepal",         "v_0": 12.9,  "k_j": 1452.0, "units": "metric",
         "outlier": True,
         "note": "k_j extremely high (~2338 veh/mi after conversion); likely mixed-traffic artifact"},
    ],
    "underwood": [
        {"source": "Drake_1967_Eisenhower",   "v_f": 76.8,  "k_0": 56.9,  "units": "imperial"},
        {"source": "Lu_Meng_2013_Beijing",     "v_f": 85.6,  "k_0": 57.5,  "units": "metric"},
        {"source": "Lu_Meng_2013_JJT",         "v_f": 134.3, "k_0": 29.0,  "units": "metric"},
        {"source": "Romanowska_2021_S6",        "v_f": 141.0, "k_0": 79.0,  "units": "metric"},
        {"source": "Tiwari_2014_Nepal",         "v_f": 64.9,  "k_0": 118.0, "units": "metric"},
    ],
}

# SR-210 engineering estimate ranges (HCM Chapter 15 + German mountain-road research)
SR210_ENGINEERING = {
    "v_f_mph":       (35.0, 50.0),   # free-flow speed
    "k_j_vehmi":     (100.0, 160.0), # jam density
    "capacity_vehhr":(600.0, 1000.0),# one-directional capacity
    "k_0_vehmi":     (15.0, 30.0),   # optimum density (Underwood)
    "v_0_mph":       (20.0, 35.0),   # speed at max flow (Greenberg)
}
print("Literature registry defined.")

In [ ]:
def convert_to_imperial(entry):
    """Return a copy of entry with speed in mph and density in veh/mi."""
    e = dict(entry)
    if e.get("units") == "metric":
        for key in ("v_f", "v_0"):
            if key in e:
                e[key] = kmh_to_mph(e[key])
        for key in ("k_j", "k_0"):
            if key in e:
                e[key] = vehkm_to_vehmi(e[key])
        e["units"] = "imperial_converted"
    return e


# Convert all entries
LIT = {
    model: [convert_to_imperial(e) for e in entries]
    for model, entries in LITERATURE_RAW.items()
}

# Display summary tables
for model_name, entries in LIT.items():
    rows = []
    for e in entries:
        row = {"Source": e["source"]}
        for p in ("v_f", "v_0", "k_j", "k_0"):
            if p in e:
                row[p] = round(e[p], 2)
        if e.get("outlier"):
            row["note"] = "OUTLIER"
        rows.append(row)
    df = pd.DataFrame(rows)
    print(f"\n{'='*60}")
    print(f"  {model_name.upper()}  (all values in mph / veh/mi)")
    print(f"{'='*60}")
    print(df.to_string(index=False))

---
## 4. Sensitivity Analysis

Before committing to the literature parameter ranges for benchmarking, it is important to understand how the shape of each model responds to parameter variation. Each figure below holds one parameter fixed at its mid-range value and sweeps the other parameter across a physically plausible range. **Color encodes the swept parameter magnitude** (viridis scale).

These plots answer questions like: *Does increasing $k_j$ in Greenshields primarily shift the jam density or also the speed-at-capacity? How does changing $k_0$ in Underwood alter the shape of speed decline at low vs. high densities?* Identifying the operating regime of SR-210 (estimated low-to-moderate density, $k < 30$ veh/mi for most hours) helps focus on the parameter region that matters most for model validation.

In [ ]:
# --- sensitivity parameter grids ---
GS_VF_GRID  = np.linspace(30, 70, 7)    # mph
GS_KJ_GRID  = np.linspace(80, 170, 7)   # veh/mi
GS_KJ_FIXED = 125.0
GS_VF_FIXED = 45.0

GR_V0_GRID  = np.linspace(15, 50, 7)    # mph
GR_KJ_GRID  = np.linspace(90, 200, 7)   # veh/mi
GR_KJ_FIXED = 130.0
GR_V0_FIXED = 28.0

UW_VF_GRID  = np.linspace(35, 90, 7)    # mph
UW_K0_GRID  = np.linspace(10, 80, 7)    # veh/mi
UW_K0_FIXED = 25.0
UW_VF_FIXED = 50.0


def sensitivity_plot_1param(ax, model_fn, param_grid, fixed_kwargs,
                             swept_param_name, param_units="",
                             cmap_name="viridis", sr210_range=None,
                             xlabel="Density k (veh/mi)",
                             ylabel="Speed v (mph)", title=""):
    """Plot speed-density curves sweeping one parameter."""
    cmap = plt.get_cmap(cmap_name, len(param_grid))
    for i, val in enumerate(param_grid):
        kwargs = {**fixed_kwargs, swept_param_name: val}
        v = model_fn(K_RANGE, **kwargs)
        v = np.clip(v, 0, None)
        ax.plot(K_RANGE, v, color=cmap(i / max(len(param_grid) - 1, 1)),
                lw=1.8, label=f"{val:.0f}{param_units}")
    if sr210_range:
        lo, hi = sr210_range
        ax.axhspan(lo, hi, alpha=0.08, color="orange",
                   label="SR-210 v_f est. (35–50 mph)")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=7, loc="upper right", title=f"{swept_param_name}",
              title_fontsize=8)
    ax.grid(True, alpha=GRID_ALPHA)
    ax.set_ylim(0, 95)
    ax.set_xlim(0, K_RANGE.max())

print("Sensitivity helpers defined.")

In [ ]:
# --- Figure: Greenshields sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
sensitivity_plot_1param(
    axes[0], greenshields,
    GS_VF_GRID, {"k_j": GS_KJ_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="viridis", sr210_range=(35, 50),
    title=f"Greenshields: v_f sweep  (k_j = {GS_KJ_FIXED:.0f} veh/mi fixed)"
)
sensitivity_plot_1param(
    axes[1], greenshields,
    GS_KJ_GRID, {"v_f": GS_VF_FIXED},
    swept_param_name="k_j", param_units=" veh/mi",
    cmap_name="plasma", sr210_range=(35, 50),
    title=f"Greenshields: k_j sweep  (v_f = {GS_VF_FIXED:.0f} mph fixed)"
)
plt.suptitle("Greenshields (1935) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_greenshields.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Greenberg sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
sensitivity_plot_1param(
    axes[0], greenberg,
    GR_V0_GRID, {"k_j": GR_KJ_FIXED},
    swept_param_name="v_0", param_units=" mph",
    cmap_name="viridis",
    title=f"Greenberg: v_0 sweep  (k_j = {GR_KJ_FIXED:.0f} veh/mi fixed)"
)
axes[0].set_ylim(0, 110)   # Greenberg can be large at low k
sensitivity_plot_1param(
    axes[1], greenberg,
    GR_KJ_GRID, {"v_0": GR_V0_FIXED},
    swept_param_name="k_j", param_units=" veh/mi",
    cmap_name="plasma",
    title=f"Greenberg: k_j sweep  (v_0 = {GR_V0_FIXED:.0f} mph fixed)"
)
axes[1].set_ylim(0, 110)
plt.suptitle("Greenberg (1959) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_greenberg.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Underwood sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
sensitivity_plot_1param(
    axes[0], underwood,
    UW_VF_GRID, {"k_0": UW_K0_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="viridis", sr210_range=(35, 50),
    title=f"Underwood: v_f sweep  (k_0 = {UW_K0_FIXED:.0f} veh/mi fixed)"
)
sensitivity_plot_1param(
    axes[1], underwood,
    UW_K0_GRID, {"v_f": UW_VF_FIXED},
    swept_param_name="k_0", param_units=" veh/mi",
    cmap_name="plasma", sr210_range=(35, 50),
    title=f"Underwood: k_0 sweep  (v_f = {UW_VF_FIXED:.0f} mph fixed)"
)
plt.suptitle("Underwood (1961) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_underwood.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Combined sensitivity (v_f sweep for each model, side by side) ---
fig, axes = plt.subplots(1, 3, figsize=FIG_W_TRIPLE, sharey=True)

sensitivity_plot_1param(
    axes[0], greenshields,
    GS_VF_GRID, {"k_j": GS_KJ_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="Blues_r", sr210_range=(35, 50),
    title="Greenshields"
)
sensitivity_plot_1param(
    axes[1], greenberg,
    GR_V0_GRID, {"k_j": GR_KJ_FIXED},
    swept_param_name="v_0", param_units=" mph",
    cmap_name="Reds_r",
    ylabel="",
    title="Greenberg"
)
sensitivity_plot_1param(
    axes[2], underwood,
    UW_VF_GRID, {"k_0": UW_K0_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="Greens_r", sr210_range=(35, 50),
    ylabel="",
    title="Underwood"
)
axes[1].set_ylim(0, 95)

plt.suptitle("Model Shape Comparison — Free-Flow Speed Sweep (v_f / v_0)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_combined.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Literature CI Bands (Monte Carlo)

These figures are the primary visualization output of the notebook. Rather than plotting a single "best estimate" curve from any one study, we treat the spread of published parameter values as **genuine uncertainty** about what parameters apply to a facility like SR-210. This is methodologically appropriate because none of the five studies was calibrated on a mountain two-lane highway — each represents an analogical reference, not a direct measurement.

**Methodology:** For each parameter, we fit a Gaussian distribution to the cross-study estimates (excluding the Tiwari Nepal Greenberg $k_j$ outlier). We then draw $N = 2000$ samples from the joint parameter distribution, compute a speed-density curve for each sample, and take the 2.5th and 97.5th percentiles of this family as the 95% CI band. The mean curve reflects the literature consensus. Individual study curves appear as thin dashed lines for transparency.

Three sets of fundamental diagrams are produced: **speed-density** ($k$–$v$), **flow-density** ($k$–$q$), and **speed-flow** ($v$–$q$). Each triple panel shows Greenshields, Greenberg, and Underwood side by side for direct comparison.

In [ ]:
def estimate_gaussian_params(values, exclude_outliers_z=2.5):
    """Estimate mean and sigma for MC sampling from a list of literature estimates.
    Optionally excludes values beyond z-score threshold.
    Returns (mu, sigma). Enforces minimum 5% CV.
    """
    arr = np.array(values, dtype=float)
    if len(arr) < 2:
        mu    = arr[0]
        sigma = mu * 0.10
        return mu, sigma
    mu    = np.mean(arr)
    sigma = np.std(arr, ddof=1)
    if exclude_outliers_z and sigma > 0:
        z   = np.abs((arr - mu) / sigma)
        arr = arr[z <= exclude_outliers_z]
        mu  = np.mean(arr)
        sigma = np.std(arr, ddof=1) if len(arr) > 1 else mu * 0.10
    sigma = max(sigma, mu * 0.05)   # minimum 5% CV prevents degenerate zero-width bands
    return mu, sigma


def mc_ci_band(model_fn, param_distributions, k_range, n_mc=N_MC, seed=42):
    """Monte Carlo CI band.

    param_distributions: list of (mu, sigma) tuples for each positional
                         parameter of model_fn (after k).
    Returns: (v_mean, v_lo, v_hi) each shape (len(k_range),)
    """
    rng    = np.random.default_rng(seed)
    curves = np.zeros((n_mc, len(k_range)))
    for i in range(n_mc):
        params = [max(rng.normal(mu, sig), 0.1) for mu, sig in param_distributions]
        v = model_fn(k_range, *params)
        curves[i] = np.clip(v, 0, None)
    v_mean = np.mean(curves, axis=0)
    v_lo   = np.percentile(curves, CI_LO, axis=0)
    v_hi   = np.percentile(curves, CI_HI, axis=0)
    return v_mean, v_lo, v_hi


# --- Gaussian parameter estimates ---
# Greenshields
GS_MU_VF, GS_SIG_VF = estimate_gaussian_params([e["v_f"] for e in LIT["greenshields"]])
GS_MU_KJ, GS_SIG_KJ = estimate_gaussian_params([e["k_j"] for e in LIT["greenshields"]])

# Greenberg — exclude Tiwari Nepal k_j outlier (k_j > 500 veh/mi)
GR_MU_V0, GR_SIG_V0 = estimate_gaussian_params([e["v_0"] for e in LIT["greenberg"]])
GR_MU_KJ, GR_SIG_KJ = estimate_gaussian_params(
    [e["k_j"] for e in LIT["greenberg"] if e["k_j"] < 500.0]
)

# Underwood
UW_MU_VF, UW_SIG_VF = estimate_gaussian_params([e["v_f"] for e in LIT["underwood"]])
UW_MU_K0, UW_SIG_K0 = estimate_gaussian_params([e["k_0"] for e in LIT["underwood"]])

print("Parameter distributions (mph / veh/mi):")
print(f"  Greenshields:  v_f ~ N({GS_MU_VF:.1f}, {GS_SIG_VF:.1f})   k_j ~ N({GS_MU_KJ:.1f}, {GS_SIG_KJ:.1f})")
print(f"  Greenberg:     v_0 ~ N({GR_MU_V0:.1f}, {GR_SIG_V0:.1f})   k_j ~ N({GR_MU_KJ:.1f}, {GR_SIG_KJ:.1f})")
print(f"  Underwood:     v_f ~ N({UW_MU_VF:.1f}, {UW_SIG_VF:.1f})   k_0 ~ N({UW_MU_K0:.1f}, {UW_SIG_K0:.1f})")

In [ ]:
# --- Compute all three CI bands ---
gs_mean, gs_lo, gs_hi = mc_ci_band(
    greenshields, [(GS_MU_VF, GS_SIG_VF), (GS_MU_KJ, GS_SIG_KJ)], K_PLOT
)
gr_mean, gr_lo, gr_hi = mc_ci_band(
    greenberg, [(GR_MU_V0, GR_SIG_V0), (GR_MU_KJ, GR_SIG_KJ)], K_PLOT
)
uw_mean, uw_lo, uw_hi = mc_ci_band(
    underwood, [(UW_MU_VF, UW_SIG_VF), (UW_MU_K0, UW_SIG_K0)], K_PLOT
)

# --- Derived capacity summary table ---
cap_rows = []
for e in LIT["greenshields"]:
    k_c, q_max = greenshields_critical(e["v_f"], e["k_j"])
    cap_rows.append({"Model": "Greenshields", "Source": e["source"],
                     "v_f (mph)": round(e["v_f"], 1), "k_j (veh/mi)": round(e["k_j"], 1),
                     "k_c (veh/mi)": round(k_c, 1), "q_max (veh/hr)": round(q_max, 0)})
for e in LIT["greenberg"]:
    k_c, q_max = greenberg_critical(e["v_0"], e["k_j"])
    outlier = " *" if e.get("outlier") else ""
    cap_rows.append({"Model": "Greenberg", "Source": e["source"] + outlier,
                     "v_f (mph)": round(e["v_0"], 1), "k_j (veh/mi)": round(e["k_j"], 1),
                     "k_c (veh/mi)": round(k_c, 1), "q_max (veh/hr)": round(q_max, 0)})
for e in LIT["underwood"]:
    k_c, q_max = underwood_critical(e["v_f"], e["k_0"])
    cap_rows.append({"Model": "Underwood", "Source": e["source"],
                     "v_f (mph)": round(e["v_f"], 1), "k_j (veh/mi)": round(e["k_0"], 1),
                     "k_c (veh/mi)": round(k_c, 1), "q_max (veh/hr)": round(q_max, 0)})

cap_df = pd.DataFrame(cap_rows)
print("\nDerived capacity summary (* = outlier excluded from CI):")
print(cap_df.rename(columns={"v_f (mph)": "param1 (mph)",
                               "k_j (veh/mi)": "param2 (veh/mi)"}).to_string(index=False))

In [ ]:
# --- Helper: plot one k-v CI panel ---
def plot_kv_panel(ax, k, v_mean, v_lo, v_hi, lit_entries,
                  speed_param_key, density_param_key,
                  model_fn, color, model_label,
                  sr210_vf_range=None, ylim=(0, 95)):
    ax.fill_between(k, v_lo, v_hi, alpha=CI_ALPHA, color=color,
                    label="95% CI (lit. uncertainty)")
    ax.plot(k, v_mean, color=color, lw=2.2, label=f"Mean curve")
    for e in lit_entries:
        if e.get("outlier"):
            continue
        p1 = e[speed_param_key]
        p2 = e[density_param_key]
        v_study = np.clip(model_fn(k, p1, p2), 0, None)
        label_txt = e["source"].replace("_", " ")
        ax.plot(k, v_study, "--", alpha=0.45, lw=1.0, color=color, label=label_txt)
    if sr210_vf_range:
        ax.axhspan(*sr210_vf_range, alpha=0.10, color="orange",
                   label="SR-210 v_f est. (35–50 mph)")
    ax.set_xlabel("Density k (veh/mi)", fontsize=10)
    ax.set_ylabel("Speed v (mph)", fontsize=10)
    ax.set_title(model_label, fontsize=11, fontweight="bold")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=GRID_ALPHA)
    ax.set_xlim(0, k.max())
    ax.set_ylim(*ylim)


# --- Figure: k-v with CI bands ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

plot_kv_panel(
    axes[0], K_PLOT, gs_mean, gs_lo, gs_hi,
    LIT["greenshields"], "v_f", "k_j", greenshields,
    "#2196F3", "Greenshields (1935)", sr210_vf_range=(35, 50)
)
plot_kv_panel(
    axes[1], K_PLOT, gr_mean, gr_lo, gr_hi,
    LIT["greenberg"], "v_0", "k_j", greenberg,
    "#E91E63", "Greenberg (1959)", ylim=(0, 110)
)
axes[1].set_ylabel("")
plot_kv_panel(
    axes[2], K_PLOT, uw_mean, uw_lo, uw_hi,
    LIT["underwood"], "v_f", "k_0", underwood,
    "#4CAF50", "Underwood (1961)", sr210_vf_range=(35, 50)
)
axes[2].set_ylabel("")

plt.suptitle(
    "Speed-Density (k–v) Fundamental Diagram\nLiterature Parameter Uncertainty — 95% CI",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("notebooks/figures/kv_lit_ci_bands.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: q-k (flow-density) with CI bands ---
gs_q_mean = K_PLOT * gs_mean
gs_q_lo   = K_PLOT * gs_lo
gs_q_hi   = K_PLOT * gs_hi

gr_q_mean = K_PLOT * gr_mean
gr_q_lo   = K_PLOT * gr_lo
gr_q_hi   = K_PLOT * gr_hi

uw_q_mean = K_PLOT * uw_mean
uw_q_lo   = K_PLOT * uw_lo
uw_q_hi   = K_PLOT * uw_hi

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, (v_mean, q_mean, q_lo, q_hi), lit_entries, (sp_key, dens_key), model_fn, color, label, crit_fn in [
    (axes[0], (gs_mean, gs_q_mean, gs_q_lo, gs_q_hi), LIT["greenshields"],
     ("v_f","k_j"), greenshields, "#2196F3", "Greenshields (1935)", greenshields_critical),
    (axes[1], (gr_mean, gr_q_mean, gr_q_lo, gr_q_hi), LIT["greenberg"],
     ("v_0","k_j"), greenberg, "#E91E63", "Greenberg (1959)", greenberg_critical),
    (axes[2], (uw_mean, uw_q_mean, uw_q_lo, uw_q_hi), LIT["underwood"],
     ("v_f","k_0"), underwood, "#4CAF50", "Underwood (1961)", underwood_critical),
]:
    ax.fill_between(K_PLOT, q_lo, q_hi, alpha=CI_ALPHA, color=color, label="95% CI")
    ax.plot(K_PLOT, q_mean, color=color, lw=2.2, label="Mean q(k)")
    for e in lit_entries:
        if e.get("outlier"):
            continue
        p1, p2 = e[sp_key], e[dens_key]
        v_s = np.clip(model_fn(K_PLOT, p1, p2), 0, None)
        q_s = K_PLOT * v_s
        ax.plot(K_PLOT, q_s, "--", alpha=0.4, lw=1.0, color=color)
    # Mark mean capacity
    k_c, q_max = crit_fn(*(GS_MU_VF, GS_MU_KJ) if label == "Greenshields (1935)"
                         else (GR_MU_V0, GR_MU_KJ) if label == "Greenberg (1959)"
                         else (UW_MU_VF, UW_MU_K0))
    ax.axvline(k_c, color=color, lw=1.0, ls=":", alpha=0.7,
               label=f"k_c = {k_c:.0f} veh/mi")
    ax.axhline(q_max, color=color, lw=1.0, ls=":", alpha=0.7,
               label=f"q_max = {q_max:.0f} veh/hr")
    ax.axhspan(*SR210_ENGINEERING["capacity_vehhr"], alpha=0.08, color="orange",
               label="SR-210 capacity est.")
    ax.set_xlabel("Density k (veh/mi)", fontsize=10)
    ax.set_ylabel("Flow q (veh/hr)", fontsize=10) if ax is axes[0] else None
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=GRID_ALPHA)
    ax.set_xlim(0, K_PLOT.max())
    ax.set_ylim(0, None)

plt.suptitle(
    "Flow-Density (k–q) Fundamental Diagram\nLiterature Parameter Uncertainty — 95% CI",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("notebooks/figures/qk_lit_ci_bands.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: v-q (speed-flow) with CI bands ---
# For v-q we plot v (y) vs q=k*v (x), parameterized by k
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, (v_mean, q_mean, v_lo, v_hi, q_lo, q_hi), color, label in [
    (axes[0], (gs_mean, gs_q_mean, gs_lo, gs_hi, gs_q_lo, gs_q_hi), "#2196F3", "Greenshields (1935)"),
    (axes[1], (gr_mean, gr_q_mean, gr_lo, gr_hi, gr_q_lo, gr_q_hi), "#E91E63", "Greenberg (1959)"),
    (axes[2], (uw_mean, uw_q_mean, uw_lo, uw_hi, uw_q_lo, uw_q_hi), "#4CAF50", "Underwood (1961)"),
]:
    # Sort by q for clean plotting (Greenshields is backward-bending, others are monotone in upper branch)
    sort_idx = np.argsort(q_mean)
    ax.fill_betweenx(v_mean[sort_idx], q_lo[sort_idx], q_hi[sort_idx],
                     alpha=CI_ALPHA, color=color, label="95% CI")
    ax.plot(q_mean[sort_idx], v_mean[sort_idx], color=color, lw=2.2, label="Mean")
    ax.axhspan(*SR210_ENGINEERING["v_f_mph"], alpha=0.08, color="orange",
               label="SR-210 v_f est.")
    ax.axvspan(*SR210_ENGINEERING["capacity_vehhr"], alpha=0.06, color="purple",
               label="SR-210 capacity est.")
    ax.set_xlabel("Flow q (veh/hr)", fontsize=10)
    ax.set_ylabel("Speed v (mph)", fontsize=10) if ax is axes[0] else None
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=GRID_ALPHA)
    ax.set_ylim(0, 95)
    ax.set_xlim(left=0)

plt.suptitle(
    "Speed-Flow (v–q) Fundamental Diagram\nLiterature Parameter Uncertainty — 95% CI",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("notebooks/figures/vq_lit_ci_bands.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. SR-210 Engineering Overlay

This figure contextualizes the literature benchmarks against SR-210's specific operating regime. Two engineering estimate ranges are overlaid:

- **Orange band** (horizontal, 35–50 mph): Expected free-flow speed range, derived from HCM Chapter 15 mountainous terrain methodology and posted speed limits on SR-210 (25–50 mph). Grades up to 11% depress actual free-flow speed well below posted limits on the steepest sections.
- **Purple band** (vertical, 100–160 veh/mi): Expected jam density range, derived from German mountain-road research and HCM two-lane highway methodology. Substantially lower than freeway jam densities (185–250 veh/mi) due to narrower lane widths, sharp curves, and no-passing constraints.

A horizontal dashed line marks the SR-210 one-directional capacity midpoint (~800 veh/hr). The intersection of these bands with the three model mean curves identifies the expected free-flow operating point and helps select which model's parameterization is most physically defensible for SR-210 validation.

In [ ]:
fig, ax = plt.subplots(figsize=FIG_W_WIDE)

# CI bands
ax.fill_between(K_PLOT, gs_lo, gs_hi, alpha=CI_ALPHA, color="#2196F3")
ax.fill_between(K_PLOT, gr_lo, gr_hi, alpha=CI_ALPHA, color="#E91E63")
ax.fill_between(K_PLOT, uw_lo, uw_hi, alpha=CI_ALPHA, color="#4CAF50")

# Mean curves
ax.plot(K_PLOT, gs_mean, color="#2196F3", lw=2.2, label="Greenshields (1935) mean")
ax.plot(K_PLOT, gr_mean, color="#E91E63", lw=2.2, label="Greenberg (1959) mean")
ax.plot(K_PLOT, uw_mean, color="#4CAF50", lw=2.2, label="Underwood (1961) mean")

# SR-210 engineering bands
vf_lo, vf_hi = SR210_ENGINEERING["v_f_mph"]
kj_lo, kj_hi = SR210_ENGINEERING["k_j_vehmi"]
cap_mid = sum(SR210_ENGINEERING["capacity_vehhr"]) / 2

ax.axhspan(vf_lo, vf_hi, alpha=0.12, color="orange",
           label=f"SR-210 v_f estimate ({vf_lo:.0f}–{vf_hi:.0f} mph)")
ax.axvspan(kj_lo, kj_hi, alpha=0.08, color="purple",
           label=f"SR-210 k_j estimate ({kj_lo:.0f}–{kj_hi:.0f} veh/mi)")

ax.set_xlim(0, K_PLOT.max())
ax.set_ylim(0, 95)
ax.set_xlabel("Density k (veh/mi)", fontsize=11)
ax.set_ylabel("Speed v (mph)", fontsize=11)
ax.set_title(
    "All Three Models: Literature Means + 95% CI Bands\nvs. SR-210 Engineering Estimate Ranges",
    fontsize=12, fontweight="bold"
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=GRID_ALPHA)

# Annotate SR-210 regions
ax.text(kj_lo + 1.5, 2, "SR-210 jam density range",
        fontsize=8, color="purple", alpha=0.8, rotation=90, va="bottom")
ax.text(1, vf_lo + 0.5, "SR-210 free-flow speed range",
        fontsize=8, color="darkorange", alpha=0.9)

plt.tight_layout()
plt.savefig("notebooks/figures/sr210_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. ABM Comparison Framework (Edie's Definitions)

This section provides the comparison harness for once simulation trajectory data is available. It extracts the macroscopic fundamental diagram from agent-level trajectories using **Edie's (1963) generalized definitions** — the only method that guarantees the fundamental identity $q = kv$ holds exactly by construction. Unlike virtual-loop-detector methods, Edie's spatial approach correctly captures both the free-flow and congested branches of the fundamental diagram from trajectory data.

**Edie's definitions** for a time-space cell $\Delta T \times \Delta X$:
$$k = \frac{\sum t_i}{\Delta T \cdot \Delta X}, \quad q = \frac{\sum d_i}{\Delta T \cdot \Delta X}, \quad v = \frac{q}{k} = \frac{\sum d_i}{\sum t_i}$$
where $t_i$ = time vehicle $i$ spent in the cell (seconds), $d_i$ = distance vehicle $i$ traveled in the cell (meters).

The Tier 2 collector records vehicle state every 10 steps (10 seconds by default). Each row therefore contributes $t_i = 10$ seconds and $d_i = $ `dist_step_m` (incremental distance) to its spatial cell. The `distance_traveled` column is cumulative arc-length in meters, serving as the 1D road coordinate for spatial cell assignment — no (x,y) projection required.

The three models are then fit to the extracted $(k, v)$ observations via nonlinear least squares and their parameters compared against the literature ranges from Section 5.

In [ ]:
# === 7.1  Load Tier 2 data ===
# Set SEASON_ID to your simulation run. The tier2 parquet must have been saved from
# the HybridDataCollector.  Add this after a run:
#   model.datacollector.get_tier2_dataframe().to_parquet(
#       f"data/season_outputs/{SEASON_ID}/tier2_spatial.parquet")

SEASON_ID   = "your_season_id_here"   # <-- SET THIS
TIER2_PATH  = Path(f"data/season_outputs/{SEASON_ID}/tier2_spatial.parquet")
ROAD_LENGTH_M = 19950.0   # ~12.4 miles in meters
TIER2_SAMPLE_INTERVAL = 10  # steps between Tier 2 records (default in HybridDataCollector)

tier2_df = None
if TIER2_PATH.exists():
    tier2_df = pd.read_parquet(TIER2_PATH)
    print(f"Loaded Tier 2 data: {len(tier2_df):,} records")
    print(f"  Steps:   {tier2_df['Step'].min()} – {tier2_df['Step'].max()}")
    print(f"  Agents:  {tier2_df['AgentID'].nunique():,} unique")
    print(f"  Columns: {list(tier2_df.columns)}")
else:
    print(f"No Tier 2 data found at: {TIER2_PATH}")
    print()
    print("To generate it, run the simulation and then execute:")
    print("  model.datacollector.get_tier2_dataframe().to_parquet(")
    print(f"      'data/season_outputs/{SEASON_ID}/tier2_spatial.parquet')")
    print()
    print("Sections 7.2–7.7 will be skipped until data is available.")

In [ ]:
# === 7.2  Preprocess Tier 2 ===
if tier2_df is not None:
    import ast
    df = tier2_df.copy()

    # Parse pos tuples
    sample_pos = df["pos"].iloc[0]
    if isinstance(sample_pos, str):
        df[["pos_x", "pos_y"]] = pd.DataFrame(
            df["pos"].apply(ast.literal_eval).tolist(), index=df.index
        )
    elif isinstance(sample_pos, (tuple, list)):
        df["pos_x"] = df["pos"].apply(lambda p: p[0])
        df["pos_y"] = df["pos"].apply(lambda p: p[1])

    # Compute per-step incremental distance (cumulative → incremental)
    df = df.sort_values(["AgentID", "Step"]).reset_index(drop=True)
    df["dist_step_m"] = (
        df.groupby("AgentID")["distance_traveled"]
        .diff()
        .fillna(0)
        .clip(lower=0)   # guard against negative diffs from agent respawn
    )

    print("Preprocessing complete.")
    print(f"  distance_traveled range: {df.distance_traveled.min():.0f} – {df.distance_traveled.max():.0f} m")
    print(f"  dist_step_m (per 10-step sample) stats:")
    print(f"    mean = {df.dist_step_m.mean():.1f} m,  max = {df.dist_step_m.max():.1f} m")
    print(f"  Note: each row = {TIER2_SAMPLE_INTERVAL} seconds of occupancy time")
    tier2_proc = df
else:
    tier2_proc = None

In [ ]:
# === 7.3–7.4  Edie's k/v/q computation ===

def compute_edie_kvq(df, road_length_m, n_spatial=20, n_temporal=30,
                     tier2_sample_interval=10):
    """Compute Edie (1963) k, q, v for a uniform time-space grid.

    Each row in df represents tier2_sample_interval seconds of data for one vehicle.
    Spatial bins use distance_traveled (cumulative meters) as the 1D road coordinate.

    Returns DataFrame with columns:
        x_mid_m, t_mid_step, k_vehmi, v_mph, q_vehhr, n_records
    """
    # Define cell edges
    x_edges = np.linspace(0, road_length_m, n_spatial + 1)
    t_min   = int(df["Step"].min())
    t_max   = int(df["Step"].max())
    t_edges = np.linspace(t_min, t_max, n_temporal + 1)

    dx_m    = x_edges[1] - x_edges[0]                               # meters
    dt_sec  = (t_edges[1] - t_edges[0]) * tier2_sample_interval     # seconds
    area    = dx_m * dt_sec                                          # m·s

    # Assign each record to spatial and temporal bins
    work = df.copy()
    work["x_bin"] = pd.cut(work["distance_traveled"], bins=x_edges,
                           labels=False, include_lowest=True)
    work["t_bin"] = pd.cut(work["Step"], bins=t_edges,
                           labels=False, include_lowest=True)
    work = work.dropna(subset=["x_bin", "t_bin"])
    work["x_bin"] = work["x_bin"].astype(int)
    work["t_bin"] = work["t_bin"].astype(int)

    # t_i = time each record represents (constant: sample_interval seconds)
    work["t_i"] = float(tier2_sample_interval)

    # Aggregate
    grouped = work.groupby(["x_bin", "t_bin"], observed=True)
    stats = grouped.agg(
        sum_ti=("t_i",        "sum"),   # total vehicle-seconds
        sum_di=("dist_step_m","sum"),   # total vehicle-meters
        n_records=("AgentID", "count"),
    ).reset_index()

    # Edie's definitions
    stats["k_vehm"]   = stats["sum_ti"]  / area                     # veh/m
    stats["q_vehsec"] = stats["sum_di"]  / area                     # veh/s
    stats["v_mps"]    = (stats["sum_di"] / stats["sum_ti"]).fillna(0)

    # Convert to road-standard units
    stats["k_vehmi"] = stats["k_vehm"]   * METERS_PER_MILE
    stats["q_vehhr"] = stats["q_vehsec"] * 3600.0
    stats["v_mph"]   = stats["v_mps"]    * (3600.0 / METERS_PER_MILE)

    # Cell midpoints
    stats["x_mid_m"]   = x_edges[stats["x_bin"]] + dx_m / 2
    stats["t_mid_step"] = (t_edges[stats["t_bin"].values].astype(int)
                           + int((t_edges[1] - t_edges[0]) / 2))

    # Filter near-empty cells (at least 3 vehicle records for meaningful stats)
    stats = stats[stats["n_records"] >= 3].copy()
    return stats


if tier2_proc is not None:
    edie_df = compute_edie_kvq(
        tier2_proc, ROAD_LENGTH_M,
        n_spatial=20, n_temporal=30,
        tier2_sample_interval=TIER2_SAMPLE_INTERVAL
    )
    print(f"Edie cells computed: {len(edie_df):,} non-empty cells")
    print(f"  k range: {edie_df.k_vehmi.min():.2f} – {edie_df.k_vehmi.max():.2f} veh/mi")
    print(f"  v range: {edie_df.v_mph.min():.1f}   – {edie_df.v_mph.max():.1f} mph")
    print(f"  q range: {edie_df.q_vehhr.min():.0f}   – {edie_df.q_vehhr.max():.0f} veh/hr")
else:
    edie_df = None

In [ ]:
# === 7.5  NLS model fitting ===

def fit_model(model_fn, k_obs, v_obs, p0, bounds=(0, np.inf), model_name=""):
    """Fit model_fn(k, *params) = v via nonlinear least squares.

    Returns dict with: params, pcov, rmse, r2, n_obs, converged.
    """
    mask = (k_obs > 0) & (v_obs >= 0) & np.isfinite(k_obs) & np.isfinite(v_obs)
    k_fit, v_fit = k_obs[mask], v_obs[mask]

    if len(k_fit) < len(p0) + 2:
        return {"converged": False, "model": model_name, "n_obs": len(k_fit),
                "error": "insufficient data"}
    try:
        popt, pcov = curve_fit(
            model_fn, k_fit, v_fit, p0=p0, bounds=bounds, maxfev=8000
        )
        v_pred    = model_fn(k_fit, *popt)
        residuals = v_fit - v_pred
        ss_res    = np.sum(residuals**2)
        ss_tot    = np.sum((v_fit - v_fit.mean())**2)
        rmse      = np.sqrt(np.mean(residuals**2))
        r2        = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        return {
            "model": model_name,
            "params": popt,
            "pcov":   pcov,
            "rmse":   rmse,
            "r2":     r2,
            "n_obs":  len(k_fit),
            "converged": True,
        }
    except (RuntimeError, ValueError) as exc:
        return {"converged": False, "model": model_name,
                "n_obs": len(k_fit), "error": str(exc)}


if edie_df is not None:
    k_obs = edie_df["k_vehmi"].values
    v_obs = edie_df["v_mph"].values

    gs_fit = fit_model(
        greenshields, k_obs, v_obs,
        p0=[v_obs.max(), k_obs.max() * 1.5],
        bounds=([0, 0], [200, 500]),
        model_name="Greenshields"
    )
    gr_fit = fit_model(
        greenberg, k_obs, v_obs,
        p0=[np.median(v_obs), k_obs.max() * 1.3],
        bounds=([0, 0], [200, 500]),
        model_name="Greenberg"
    )
    uw_fit = fit_model(
        underwood, k_obs, v_obs,
        p0=[v_obs.max(), np.median(k_obs)],
        bounds=([0, 0], [200, 500]),
        model_name="Underwood"
    )

    print("NLS fitting results:")
    for result in [gs_fit, gr_fit, uw_fit]:
        if result["converged"]:
            print(f"  {result['model']:14s}: params = {np.round(result['params'], 2)}, "
                  f"RMSE = {result['rmse']:.2f} mph,  pseudo-R² = {result['r2']:.3f}, "
                  f"N = {result['n_obs']}")
        else:
            print(f"  {result['model']:14s}: DID NOT CONVERGE — {result.get('error', '')}")
else:
    gs_fit = gr_fit = uw_fit = None
    print("Skipping NLS fitting (no ABM data loaded).")

In [ ]:
# === 7.6  Comparison table ===

def build_comparison_table(fit_results, lit_means):
    """Build a structured comparison of ABM-fitted params vs literature means.

    lit_means: dict of {model_name: {param_label: value}}
    """
    param_map = {
        "Greenshields": [("v_f (mph)",   0), ("k_j (veh/mi)", 1)],
        "Greenberg":    [("v_0 (mph)",   0), ("k_j (veh/mi)", 1)],
        "Underwood":    [("v_f (mph)",   0), ("k_0 (veh/mi)", 1)],
    }
    rows = []
    for result in fit_results:
        mname = result["model"]
        row   = {"Model": mname, "Converged": result["converged"]}
        if result["converged"]:
            for col_name, idx in param_map.get(mname, []):
                fitted_val = result["params"][idx]
                lit_val    = lit_means.get(mname, {}).get(col_name, np.nan)
                row[f"{col_name} (ABM)"]  = round(fitted_val, 2)
                row[f"{col_name} (lit)"]  = round(lit_val,    2)
                if not np.isnan(lit_val) and lit_val != 0:
                    row[f"% diff"] = round(100 * (fitted_val - lit_val) / lit_val, 1)
            row["RMSE (mph)"] = round(result["rmse"], 2)
            row["pseudo-R²"]  = round(result["r2"],   3)
            row["N cells"]    = result["n_obs"]
        rows.append(row)
    return pd.DataFrame(rows)


lit_means = {
    "Greenshields": {"v_f (mph)": round(GS_MU_VF, 1), "k_j (veh/mi)": round(GS_MU_KJ, 1)},
    "Greenberg":    {"v_0 (mph)": round(GR_MU_V0, 1), "k_j (veh/mi)": round(GR_MU_KJ, 1)},
    "Underwood":    {"v_f (mph)": round(UW_MU_VF, 1), "k_0 (veh/mi)": round(UW_MU_K0, 1)},
}

if gs_fit is not None:
    comp_df = build_comparison_table([gs_fit, gr_fit, uw_fit], lit_means)
    print("ABM vs. Literature parameter comparison:")
    print(comp_df.to_string(index=False))
else:
    print("Skipping comparison table (no ABM fit results available).")

In [ ]:
# === 7.7  Overlay: ABM Edie scatter + fitted curves + literature CI bands ===

if edie_df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    configs = [
        (axes[0], "Greenshields (1935)", "#2196F3",
         gs_mean, gs_lo, gs_hi, gs_fit, greenshields),
        (axes[1], "Greenberg (1959)",    "#E91E63",
         gr_mean, gr_lo, gr_hi, gr_fit, greenberg),
        (axes[2], "Underwood (1961)",    "#4CAF50",
         uw_mean, uw_lo, uw_hi, uw_fit, underwood),
    ]

    for ax, label, color, v_mean, v_lo, v_hi, fit_result, model_fn in configs:
        # Literature CI band
        ax.fill_between(K_PLOT, v_lo, v_hi, alpha=CI_ALPHA, color=color,
                        label="Lit. 95% CI")
        ax.plot(K_PLOT, v_mean, color=color, lw=1.5, ls="--",
                label="Lit. mean")
        # ABM Edie scatter
        ax.scatter(edie_df["k_vehmi"], edie_df["v_mph"],
                   s=14, alpha=0.5, color="#333333", zorder=5,
                   label="ABM (Edie cells)")
        # ABM fitted curve
        if fit_result and fit_result["converged"]:
            v_fitted = np.clip(model_fn(K_PLOT, *fit_result["params"]), 0, None)
            ax.plot(K_PLOT, v_fitted, color="black", lw=2.2,
                    label=f"ABM fit  R²={fit_result['r2']:.2f}")
        ax.set_xlabel("Density k (veh/mi)", fontsize=10)
        ax.set_ylabel("Speed v (mph)", fontsize=10) if ax is axes[0] else None
        ax.set_title(label, fontsize=11, fontweight="bold")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=GRID_ALPHA)
        ax.set_xlim(0, K_PLOT.max())
        ax.set_ylim(0, 95)

    plt.suptitle(
        "ABM Simulation vs. Classical Models — SR-210 Fundamental Diagram",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("notebooks/figures/abm_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Overlay figure skipped — load Tier 2 data and rerun Section 7 cells.")

---
## 8. Summary and Interpretation Guide

This notebook provides a complete validation framework for benchmarking the SR-210 Mesa ABM against macroscopic traffic flow theory. The three classical models together span the full density range through complementary strengths, and the literature CI bands in Section 5 represent an informed prior for what parameters are physically plausible on a mountain two-lane highway.

**Interpreting ABM comparison results (Section 7):**

| Observation | Likely Diagnosis |
|---|---|
| ABM $v_f$ within literature 95% CI | Free-flow behavior consistent with empirical studies |
| ABM $v_f$ substantially above literature mean | Vehicles may be exceeding speed limits; check car_agent speed cap |
| ABM $k_j$ substantially below literature range | Model may over-space vehicles; check gap-acceptance parameters |
| High RMSE (> 10 mph) for all three models | ABM does not follow any single-regime classical FD — possibly a valid finding due to SR-210's grade-induced non-equilibrium dynamics and platoon formation |
| Greenberg fits poorly at low density | Expected — Greenberg diverges at $k \to 0$; valid only for congested regime |
| Wide scatter around any fitted curve | Likely reflects multi-regime behavior (uphill vs. downhill) or hysteresis; consider fitting direction-separately |

**Model limitations reminder:**
- **Greenshields:** Linear — symmetric parabola in q-k diagram; over-simplified for congested regime. Useful as a baseline.
- **Greenberg:** Best at congestion, physically meaningless at low density. Only applicable to peak-demand ski days.
- **Underwood:** Best at free-flow (the dominant SR-210 regime). Overestimates $v_f$ and has no finite $k_j$.
- All three are **single-regime equilibrium models** — none can represent hysteresis, capacity drops, or grade-stratified behavior.

**To re-run the ABM comparison:** set `SEASON_ID` in Section 7.1, ensure `tier2_spatial.parquet` exists, and rerun all cells from Section 7 onward.

In [ ]:
# --- SR-210 engineering range summary ---
sr210_df = pd.DataFrame([
    {"Parameter": "v_f (mph)",            "SR-210 Low": 35,   "SR-210 High": 50,
     "GS lit. mean": round(GS_MU_VF, 1),  "GR lit. mean": "—",
     "UW lit. mean": round(UW_MU_VF, 1)},
    {"Parameter": "k_j (veh/mi)",         "SR-210 Low": 100,  "SR-210 High": 160,
     "GS lit. mean": round(GS_MU_KJ, 1),  "GR lit. mean": round(GR_MU_KJ, 1),
     "UW lit. mean": "—"},
    {"Parameter": "k_0 (veh/mi)",         "SR-210 Low": 15,   "SR-210 High": 30,
     "GS lit. mean": "—",                  "GR lit. mean": "—",
     "UW lit. mean": round(UW_MU_K0, 1)},
    {"Parameter": "v_0 (mph)",             "SR-210 Low": 20,   "SR-210 High": 35,
     "GS lit. mean": "—",                  "GR lit. mean": round(GR_MU_V0, 1),
     "UW lit. mean": "—"},
    {"Parameter": "Capacity (veh/hr/dir)", "SR-210 Low": 600,  "SR-210 High": 1000,
     "GS lit. mean": "—",                  "GR lit. mean": "—",
     "UW lit. mean": "—"},
])

print("SR-210 engineering estimate ranges vs. literature means:")
print(sr210_df.to_string(index=False))

print("\nFigures saved to notebooks/figures/:")
for f in sorted(Path("notebooks/figures").glob("*.png")):
    print(f"  {f.name}")